<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week6/Day5/ExerciseXP/mini_projet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mini-projet : Assistant d’analyse des sentiments avec optimisation de BERT

In [ ]:
# 1. Mise à jour de protobuf pour aligner le moteur d'exécution (Runtime) avec le Gencode
%pip install --quiet --upgrade protobuf >=5.29.6

# 2. Recharger et forcer la liaison TensorFlow complète pour Transformers
%pip install --quiet --upgrade transformers[tf] datasets tensorflow-datasets


In [ ]:
# =====================================================================
# Mini-projet : Assistant d'analyse des sentiments — BERT + PyTorch
# Compatible : Colab, transformers 5.x, zéro install supplémentaire
# =====================================================================

# ── Cellule 1 : Installation minimale ────────────────────────────────
# Une seule dépendance manquante dans votre env Colab
!pip install --quiet torch

# =====================================================================
# Cellule 2 — Imports et configuration
# =====================================================================
import os
import platform
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup

print("--- Étape 0 : Configuration ---")
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")

import transformers
print(f"Transformers : {transformers.__version__}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {DEVICE}")

# Hyperparamètres adaptés CPU/GPU
MAX_LENGTH = 128    # Réduire à 64 si très lent
BATCH_SIZE = 16     # Réduire à 8 si OOM sur GPU ou très lent sur CPU
EPOCHS     = 2
MAX_TRAIN  = 3000   # None = 25 000 exemples (plusieurs heures sur CPU)
MAX_TEST   = 600    # None = 25 000 exemples
MODEL_DIR  = "bert_sentiment_model"
os.makedirs(MODEL_DIR, exist_ok=True)

# =====================================================================
# Cellule 3 — Chargement IMDB via Keras (inclus dans TF, toujours dispo)
# =====================================================================
print("\n--- Étape 1 : Chargement IMDB via keras.datasets ---")

# keras.datasets.imdb est intégré à TensorFlow — aucune install requise
import tensorflow.keras as keras

(x_train_enc, y_train_raw), (x_test_enc, y_test_raw) = \
    keras.datasets.imdb.load_data(num_words=20000)

# Reconstruction du texte brut depuis le dictionnaire intégré
word_index    = keras.datasets.imdb.get_word_index()
index_to_word = {v + 3: k for k, v in word_index.items()}
index_to_word.update({0: "<PAD>", 1: "<START>", 2: "<UNK>", 3: "<UNUSED>"})

def decode_review(encoded) -> str:
    return " ".join(index_to_word.get(i, "<UNK>") for i in encoded)

# Sous-ensemble pour limiter le temps d'exécution sur CPU
train_texts  = [decode_review(x) for x in x_train_enc[:MAX_TRAIN]]
train_labels = y_train_raw[:MAX_TRAIN].tolist()
test_texts   = [decode_review(x) for x in x_test_enc[:MAX_TEST]]
test_labels  = y_test_raw[:MAX_TEST].tolist()

print(f"Train : {len(train_texts)} exemples | Test : {len(test_texts)} exemples")
print(f"Aperçu : \"{train_texts[0][:100]}...\"")
print(f"Label  : {train_labels[0]}  (1=positif, 0=négatif)")

# =====================================================================
# Cellule 4 — Tokenisation BERT (batch unique, pas de tf.data)
# =====================================================================
print("\n--- Étape 2 : Tokenisation BERT ---")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class IMDBDataset(Dataset):
    """
    Tokenise tout le corpus une seule fois à l'initialisation.
    Beaucoup plus rapide que tokeniser à chaque batch.
    """
    def __init__(self, texts: list, labels: list, max_len: int = MAX_LENGTH):
        print(f"  Tokenisation de {len(texts)} exemples (patientez...)  ")
        self.encodings = tokenizer(
            texts,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
        print(f"  ✅ input_ids shape : {self.encodings['input_ids'].shape}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "token_type_ids": self.encodings["token_type_ids"][idx],
            "label":          self.labels[idx],
        }

train_dataset = IMDBDataset(train_texts, train_labels)
test_dataset  = IMDBDataset(test_texts,  test_labels)

# num_workers=0 obligatoire sur Colab (évite les erreurs de fork)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=(DEVICE.type=="cuda"))
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(DEVICE.type=="cuda"))

print(f"Batches — train : {len(train_loader)} | test : {len(test_loader)}")

# =====================================================================
# Cellule 5 — Modèle BERT + optimiseur + scheduler
# =====================================================================
print("\n--- Étape 3 : Chargement de BERT ---")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
)
model.to(DEVICE)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = max(1, total_steps // 10)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    eps=1e-8,
    weight_decay=0.01,
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres totaux    : {total_params:,}")
print(f"Paramètres entraînés : {trainable_params:,}")
print(f"Steps total / warmup : {total_steps} / {warmup_steps}")

# =====================================================================
# Cellule 6 — Fonctions d'entraînement et d'évaluation
# =====================================================================
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for step, batch in enumerate(loader):
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        ttype = batch["token_type_ids"].to(DEVICE)
        labs  = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask,
                    token_type_ids=ttype, labels=labs)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += out.loss.item()
        correct    += (out.logits.argmax(-1) == labs).sum().item()
        total      += labs.size(0)

        if (step + 1) % 25 == 0:
            print(f"    Step {step+1:>3}/{len(loader)} | "
                  f"Loss: {total_loss/(step+1):.4f} | "
                  f"Acc : {correct/total:.4f}")

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            ttype = batch["token_type_ids"].to(DEVICE)
            labs  = batch["label"].to(DEVICE)

            out = model(input_ids=ids, attention_mask=mask,
                        token_type_ids=ttype, labels=labs)
            total_loss += out.loss.item()
            correct    += (out.logits.argmax(-1) == labs).sum().item()
            total      += labs.size(0)

    return total_loss / len(loader), correct / total

# =====================================================================
# Cellule 7 — Entraînement avec sauvegarde du meilleur modèle
# =====================================================================
print("\n--- Étape 4 : Fine-tuning BERT ---")
best_val_acc    = 0.0
best_model_path = os.path.join(MODEL_DIR, "best_model.pt")

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*55}")
    print(f"  ÉPOQUE {epoch}/{EPOCHS}")
    print(f"{'='*55}")

    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler)
    v_loss,  v_acc  = eval_epoch(model, test_loader)

    print(f"\n  Train → Loss : {tr_loss:.4f} | Acc : {tr_acc:.4f}")
    print(f"  Val   → Loss : {v_loss:.4f}   | Acc : {v_acc:.4f}")

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✅ Meilleur modèle sauvegardé (val_acc={best_val_acc:.4f})")

# Rechargement du meilleur checkpoint
print(f"\nRestauration du meilleur checkpoint (val_acc={best_val_acc:.4f})")
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

# =====================================================================
# Cellule 8 — Évaluation finale
# =====================================================================
print("\n--- Étape 5 : Évaluation finale ---")
test_loss, test_acc = eval_epoch(model, test_loader)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

# =====================================================================
# Cellule 9 — Assistant d'inférence réutilisable
# =====================================================================
class SentimentAssistant:
    """
    Assistant réutilisable pour l'analyse de sentiment.
    Accepte une phrase ou une liste de phrases.
    """
    LABELS = {0: "Négatif ❌", 1: "Positif ✅"}

    def __init__(self, trained_model, bert_tokenizer, max_len: int = MAX_LENGTH):
        self.model     = trained_model
        self.tokenizer = bert_tokenizer
        self.max_len   = max_len
        self.model.eval()

    def predict(self, texts):
        if isinstance(texts, str):
            texts = [texts]

        encoded = self.tokenizer(
            texts,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors="pt",
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        with torch.no_grad():
            probs = torch.softmax(
                self.model(**encoded).logits, dim=-1
            ).cpu().numpy()

        return [
            {
                "text":       text,
                "label":      self.LABELS[int(p.argmax())],
                "confidence": round(float(p.max()), 4),
                "scores": {
                    self.LABELS[0]: round(float(p[0]), 4),
                    self.LABELS[1]: round(float(p[1]), 4),
                },
            }
            for text, p in zip(texts, probs)
        ]

    def print_results(self, results: list):
        print("\n" + "=" * 65)
        print("  RÉSULTATS DE L'ASSISTANT D'ANALYSE DE SENTIMENT")
        print("=" * 65)
        for r in results:
            print(f"\n  Phrase     : {r['text']}")
            print(f"  Prédiction : {r['label']}  (confiance : {r['confidence']:.1%})")
            print(f"  Scores     : {r['scores']}")
        print("=" * 65)


print("\n--- Étape 6 : Test de l'assistant ---")
assistant = SentimentAssistant(model, tokenizer)

results = assistant.predict([
    "The onboarding emails were confusing, but the agent fixed everything politely.",
    "Absolutely terrible experience. I will never use this service again.",
    "Outstanding product quality and lightning-fast delivery. Highly recommend!",
    "It was okay, nothing special but did the job.",
])
assistant.print_results(results)